In [1]:
import time

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

I0000 00:00:1795164438.730650   39001 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795164438.732281   39001 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795164438.882882   39001 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795164444.051003   39001 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
MODEL_PATH = "face_landmarker.task"
CAMERA_ID = 0

IRIS_L = 468
IRIS_R = 473

In [3]:
def create_detector():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options =  vision.FaceLandmarkerOptions(
        base_options = base_options,
        running_mode = vision.RunningMode.VIDEO,
        num_faces = 1
    )
    return vision.FaceLandmarker.create_from_options(options)

In [4]:
def get_eyes(result, w, h):
    if not result.face_landmarks:
        return None

    points = result.face_landmarks[0]

    l = points[IRIS_L]
    r = points[IRIS_R]

    l_px = (int(l.x * w), int(l.y * h))
    r_px = (int(r.x * w), int(r.y * h))

    return l_px, r_px


In [5]:
detector = create_detector()

v = cv2.VideoCapture(CAMERA_ID)

t_0 = time.perf_counter()

while True:
    ret, frame = v.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    timestamp = int((time.perf_counter() - t_0) * 1000)
    r = detector.detect_for_video(mp_img, timestamp)

    eyes = get_eyes(r, w, h)
    if eyes is not None:
        for center in eyes:
            cv2.circle(frame, center, 6, (0, 255, 255), 2)
            cv2.circle(frame, center, 1, (0, 0, 255), -1)

        cv2.line(frame, eyes[0], eyes[1], (0, 255, 0), 1)
    else:
        cv2.putText(
            frame, "Sin cara", (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2
        )

    cv2.imshow("Filtro", frame)

    key = cv2.waitKey(1) & 0xFF
    if key in (ord("q"), 27):
            break
    
v.release()
cv2.destroyAllWindows()
detector.close()

W0000 00:00:1795164454.428479   39101 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795164454.449079   39105 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795164454.475450   39105 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
